## Process Kaitlin Naughten FESOM ocean data

Created by Yanmei Tian <yanmeiti@buffalo.edu>

This notebook contains the core processing for the two FESOM RCP8.5 datasets:

- `RCP8.5_ACCESS`
- `RCP8.5_MMM`

For each dataset, **cold** is the first 20 annual fields (2006–2025) and **warm** is the last 20 annual fields (2081–2100). Temperature, salinity, thermal forcing and basal melt products are created.

###  imports

In [ ]:
from pathlib import Path
import os
import re
import subprocess

import numpy as np
import xarray as xr
from jinja2 import Environment, FileSystemLoader, StrictUndefined

os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")

HERE = Path("/home/yanmeiti/ismip7-antarctic-ocean-forcing/parameterisations/process_data")
DATA_DIR = Path("/home/yanmeiti/ismip7-antarctic-ocean-forcing/Data")
FESOM_DIR = DATA_DIR / "ocean_data" / "Kaitlin_Naughten" / "FESOM"
TOPO_DIR = DATA_DIR / "topg"
OUTPUT_DIR = DATA_DIR / "output"
WORK_DIR = FESOM_DIR / "processing_work"
TEMPLATE = HERE / "namelisttemplate.nml"
GRID_FILE = TOPO_DIR / "ismip_8km_60m_grid.nc"
BEDMAP_FILE = TOPO_DIR / "bedmap3_ismip_8km.nc"
BASIN_FILE = TOPO_DIR / "basin_numbers_ismip8km_v2.nc"

WORK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OVERWRITE = False

In [ ]:
EXPERIMENTS = {
    "FESOM_ACCESS": {
        "directory": FESOM_DIR / "RCP8.5_ACCESS",
        "pattern": "FESOM_RCP8.5_ACCESS_*.nc",
        "output_prefix": "Naughten_FESOM_ACCESS",
    },
    "FESOM_MMM": {
        "directory": FESOM_DIR / "RCP8.5_MMM",
        "pattern": "FESOM_RCP8.5_MMM_*.nc",
        "output_prefix": "Naughten_FESOM_MMM",
    },
}


def year_from_name(path):
    match = re.search(r"_(\d{4})\.nc$", path.name)
    if match is None:
        raise ValueError(f"Cannot read year from {path.name}")
    return int(match.group(1))


def annual_files(experiment):
    files = sorted(
        experiment["directory"].glob(experiment["pattern"]),
        key=year_from_name,
    )
    years = np.array([year_from_name(path) for path in files])
    np.testing.assert_array_equal(years, np.arange(2006, 2101))
    return files


for name, experiment in EXPERIMENTS.items():
    files = annual_files(experiment)
    print(name, len(files), "files:", year_from_name(files[0]), "to", year_from_name(files[-1]))

### cold and warm climatologies


```text
temperature → theta_ocean
salinity    → salinity_ocean
z           → z_extrap
```

Floating- and grounded-ice cells are set to missing so the extrapolation tools can fill the ice-shelf cavities consistently.

In [ ]:
PERIODS = {
    "cold": np.arange(2006, 2026),
    "warm": np.arange(2081, 2101),
}


def check_output(path):
    path = Path(path)
    if path.exists() and not OVERWRITE:
        raise FileExistsError(
            f"{path} exists; set OVERWRITE=True only when replacement is intended"
        )
    path.parent.mkdir(parents=True, exist_ok=True)


def make_ocean_climatology(experiment_name, experiment, mode):
    selected_years = PERIODS[mode]
    files = [
        path for path in annual_files(experiment)
        if year_from_name(path) in selected_years
    ]
    np.testing.assert_array_equal(
        [year_from_name(path) for path in files], selected_years
    )

    output_file = WORK_DIR / f"{experiment_name}_{mode}_prepared.nc"
    check_output(output_file)

    # Each source file contains one year. Opening by coordinates is lazy.
    with xr.open_mfdataset(files, combine="by_coords", chunks={"time": 1}) as source:
        ds = source.rename({
            "temperature": "theta_ocean",
            "salinity": "salinity_ocean",
            "z": "z_extrap",
        })[["theta_ocean", "salinity_ocean"]]
        climatology = ds.mean("time", keep_attrs=True).expand_dims(time=[0.0])
        climatology.time.attrs = {
            "units": "days since 2000-01-01 00:00:00",
            "calendar": "proleptic_gregorian",
            "standard_name": "time",
            "long_name": f"dummy time for the {selected_years[0]}-{selected_years[-1]} climatology",
        }

        with xr.open_dataset(BEDMAP_FILE) as bedmap:
            open_ocean = (
                (bedmap.floating_frac == 0) &
                (bedmap.grounded_frac == 0)
            )
            climatology["theta_ocean"] = climatology.theta_ocean.where(open_ocean)
            climatology["salinity_ocean"] = climatology.salinity_ocean.where(open_ocean)

        climatology = climatology.transpose("time", "z_extrap", "y", "x")
        climatology.attrs.update({
            "title": f"Kaitlin Naughten {experiment_name} {mode} climatology",
            "history": f"mean over {selected_years[0]}-{selected_years[-1]}; ice masked",
        })
        encoding = {
            "theta_ocean": {"_FillValue": -9999.0, "dtype": "float32"},
            "salinity_ocean": {"_FillValue": -9999.0, "dtype": "float32"},
            "time": {"_FillValue": None},
        }
        climatology.to_netcdf(
            output_file, encoding=encoding, unlimited_dims="time"
        )

    print("Wrote", output_file)
    return output_file

In [ ]:
# set RUN_CLIMATOLOGIES=True to create the climatologies.
RUN_CLIMATOLOGIES = False
if RUN_CLIMATOLOGIES:
    for experiment_name, experiment in EXPERIMENTS.items():
        for mode in PERIODS:
            make_ocean_climatology(experiment_name, experiment, mode)
else:
    print("Skipped; set RUN_CLIMATOLOGIES=True to create the climatologies.")

###  Horizontal and vertical extrapolation

Temperature and salinity are extrapolated independently using the ISMIP7 tools and `namelisttemplate.nml`.

In [ ]:
FIELD_VARIABLES = {"T": "theta_ocean", "S": "salinity_ocean"}


def make_namelist(experiment_name, experiment, mode, suffix):
    prepared = WORK_DIR / f"{experiment_name}_{mode}_prepared.nc"
    horizontal = WORK_DIR / f"{experiment_name}_{mode}_{suffix}_horizontal.nc"
    extrapolated = WORK_DIR / f"{experiment_name}_{mode}_{suffix}_zextrap.nc"
    namelist = WORK_DIR / f"namelist_{experiment_name}_{mode}_{suffix}.nml"
    if not prepared.is_file():
        raise FileNotFoundError(prepared)
    check_output(horizontal)
    check_output(extrapolated)

    env = Environment(
        loader=FileSystemLoader(str(TEMPLATE.parent)),
        undefined=StrictUndefined,
        keep_trailing_newline=True,
    )
    text = env.get_template(TEMPLATE.name).render(
        file_in=str(prepared.resolve()),
        file_out_horizontal=str(horizontal.resolve()),
        file_out=str(extrapolated.resolve()),
        file_basin=str(BASIN_FILE.resolve()),
        file_topo=str(BEDMAP_FILE.resolve()),
        variable=FIELD_VARIABLES[suffix],
        z_name="z_extrap",
    )
    namelist.write_text(text.replace("\r\n", "\n"), encoding="utf-8")
    return namelist


def run_extrapolation():
    namelists = [
        make_namelist(name, experiment, mode, suffix)
        for name, experiment in EXPERIMENTS.items()
        for mode in PERIODS
        for suffix in ("T", "S")
    ]
    for namelist in namelists:
        subprocess.run(["i7aof_extrap_horizontal", str(namelist)], check=True)
    for namelist in namelists:
        subprocess.run(["i7aof_extrap_vertical", str(namelist)], check=True)

In [ ]:
# set RUN_EXTRAPOLATION=True to run the extrapolation tools.
RUN_EXTRAPOLATION = False
if RUN_EXTRAPOLATION:
    run_extrapolation()
else:
    print("Skipped; set RUN_EXTRAPOLATION=True to run the extrapolation tools.")

In [ ]:
FINAL_NAMES = {
    "T": ("theta_ocean", "thetao", "deg C", "sea water potential temperature"),
    "S": ("salinity_ocean", "so", "psu", "sea water salinity"),
}


def finalize_field(experiment_name, experiment, mode, suffix):
    input_file = WORK_DIR / f"{experiment_name}_{mode}_{suffix}_zextrap.nc"
    output_file = OUTPUT_DIR / f"{experiment['output_prefix']}_{mode}_{suffix}.nc"
    check_output(output_file)
    source_name, final_name, units, long_name = FINAL_NAMES[suffix]

    with xr.open_dataset(input_file) as source, xr.open_dataset(GRID_FILE) as grid:
        result = source[[source_name]].interp(
            z_extrap=grid.z, method="linear"
        ).squeeze("time", drop=True)
        result = result.drop_vars(["lat", "lon", "z_extrap"], errors="ignore")
        result = result.rename({source_name: final_name})
        result.encoding.pop("unlimited_dims", None)
        result[final_name].attrs = {"long_name": long_name, "units": units}
        result.to_netcdf(
            output_file,
            encoding={final_name: {"_FillValue": -9999.0, "dtype": "float32"}},
        )
    print("Wrote", output_file)
    return output_file

### Thermal forcing

Thermal forcing uses the pressure-dependent linearized freezing point from Reese et al. (2018).

In [ ]:
def calculate_thermal_forcing(experiment, mode):
    prefix = OUTPUT_DIR / f"{experiment['output_prefix']}_{mode}"
    output_file = Path(f"{prefix}_TF.nc")
    check_output(output_file)
    with xr.open_dataset(f"{prefix}_T.nc") as temperature, \
         xr.open_dataset(f"{prefix}_S.nc") as salinity:
        pressure = 1028.0 * 9.81 * temperature.z * -1.0
        freezing_point = -0.0572 * salinity.so + 0.0788 - 7.77e-8 * pressure
        result = xr.Dataset({
            "tf": (temperature.thetao - freezing_point).astype("float32")
        })
        result.tf.attrs = {"long_name": "ocean thermal forcing", "units": "deg C"}
        result.to_netcdf(
            output_file,
            encoding={"tf": {"_FillValue": -9999.0, "dtype": "float32"}},
        )
    print("Wrote", output_file)

In [ ]:
#set RUN_FINAL_OCEAN_FILES=True to write T, S and TF.
RUN_FINAL_OCEAN_FILES = False
if RUN_FINAL_OCEAN_FILES:
    for experiment_name, experiment in EXPERIMENTS.items():
        for mode in PERIODS:
            finalize_field(experiment_name, experiment, mode, "T")
            finalize_field(experiment_name, experiment, mode, "S")
            calculate_thermal_forcing(experiment, mode)
else:
    print("Skipped; set RUN_FINAL_OCEAN_FILES=True to write T, S and TF.")

### Basal melt climatologies


In [ ]:
ICE_DENSITY = 917.0


def make_melt_climatology(experiment, mode):
    selected_years = PERIODS[mode]
    files = [
        path for path in annual_files(experiment)
        if year_from_name(path) in selected_years
    ]
    output_file = OUTPUT_DIR / f"{experiment['output_prefix']}_{mode}_m.nc"
    check_output(output_file)

    with xr.open_mfdataset(files, combine="by_coords", chunks={"time": 1}) as source:
        melt = source.basal_melt * ICE_DENSITY
        mean = melt.mean("time", keep_attrs=True)
        uncertainty = melt.std("time", keep_attrs=True) / np.sqrt(len(files))
        result = xr.Dataset({
            "melt_rate": mean.astype("float32"),
            "melt_rate_uncert": uncertainty.astype("float32"),
        })
        result.melt_rate.attrs = {
            "long_name": "ice shelf basal melt rate",
            "units": "kg m-2 yr-1",
        }
        result.melt_rate_uncert.attrs = {
            "long_name": "standard error of the 20-year mean basal melt rate",
            "units": "kg m-2 yr-1",
        }
        result.to_netcdf(output_file)
    print("Wrote", output_file)

In [ ]:
#set RUN_MELT_FILES=True to write basal-melt products.
RUN_MELT_FILES = False
if RUN_MELT_FILES:
    for experiment in EXPERIMENTS.values():
        for mode in PERIODS:
            make_melt_climatology(experiment, mode)
else:
    print("Skipped; set RUN_MELT_FILES=True to write basal-melt products.")

## Outputs

For ACCESS and MMM, the notebook writes cold/warm files for temperature (`thetao`), salinity (`so`), thermal forcing (`tf`) and basal melt (`melt_rate`, `melt_rate_uncert`). 